# Missão Aurora Siger: Verificação de segurança pré-decolagem
---
**Autores:** André Raposo (RM568837), Google Gemini  

## 1. Geração de dados regulares
Inícia a geração de dados de telemetria utilizando as bibliotecas `pandas` e `numpy`. Considera um cenário ideal onde todos os dados estão dentro da faixa de limites esperada.
 

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

n_rows = 500
start_time = datetime.now()

# Gera base de dados com valores randomicos dentro do limite.
data = {
    "timestamp": [start_time + timedelta(minutes=i) for i in range(n_rows)],
    "internal_temp_c": np.random.uniform(18, 35, n_rows),
    "external_temp_c": np.random.uniform(-5, 30, n_rows),
    "battery_voltage_v": np.random.uniform(46, 52, n_rows),
    "battery_current_a": np.random.uniform(20, 120, n_rows),
    "battery_soc_percentage": np.random.uniform(60, 100, n_rows),
    "battery_capacity_ah": np.random.uniform(80, 120, n_rows),
    "power_load_kw": np.random.uniform(5, 25, n_rows),
    "energy_loss_percent": np.random.uniform(2, 8, n_rows),
    "tank_pressure_bar": np.random.uniform(95, 145, n_rows),
    "estimated_autonomy_min": np.random.uniform(45, 180, n_rows),
    "structural_integrity": 1,
    "critical_modules_status": 1,
    "telemetry_link_status": 1,
    "is_anomaly": False
}

# Cálcula energia disponível
data["energy_available_kwh"] = (data["battery_voltage_v"] * data["battery_capacity_ah"] * (data["battery_soc_percentage"]/100)) / 1000

# Carrega os dados em um dataframe
df = pd.DataFrame(data)

## 2. Geração de dados com anomalias
Introduz um grau de 5% de anomalias nos dados gerados anteriormente

---
O trecho de código foi parcialmente criado utilizando o Google Gemini com o seguinte prompt:   

> "Com base nos dados gerados anteriormente introduza anomalia nos dados de telemetria
>
> Regras: 
> - Insira anomalia em 5% dos dados 
> - Temperatura interna entre 45 e 70 
> - Nível de bateria entre 5 e 25 
> - Pressão fora da faixa (40-80 ou 160-220) 
> - Integridade estrutural 0 
> - Estado dos módulos críticos 0 
> - Estados link de telemetria 0 
>
> Saída 
> - Arquivo CSV válido"

O código foi avaliado e refatorado para remover "alucionações" e melhorar a legibilidade:   


**Alterações:**
- Um dos tipos de anomalia estava com um erro no nome
- Utiliza enum para definir tipos de anomalia `AnomalyTypes` para evitar possíveis erros de digitação.

In [12]:
# Define a porcentagem de anomalia nos dados: 5%
from enum import Enum
from typing import List


n_anomalies = int(len(df) * 0.05)

# Busca por indices aleatóriamente para introduzir anomalias
anomaly_idx = np.random.choice(df.index, n_anomalies, replace=False)


# Define os tipos de dados que poderão ter anomalias
# Define Anomaly Types using an Enum
class AnomalyTypes(Enum):
    INTERNAL_TEMP = 'internal_temp_c'
    BATTERY_SOC = 'battery_soc_percentage'
    TANK_PRESSURE = 'tank_pressure_bar'
    STRUCTURAL_INTEGRITY = 'structural_integrity'
    CRITICAL_MODULES_STATUS = 'critical_modules_status'
    TELEMETRY_LINK_STATUS = 'telemetry_link_status'

anomaly_types: List = list(AnomalyTypes)

for idx in anomaly_idx:
    df.loc[idx, "is_anomaly"] = True
    anomaly_type = np.random.choice(anomaly_types)

    match anomaly_type:
        case AnomalyTypes.INTERNAL_TEMP:
            df.loc[idx, anomaly_type.value] = np.random.uniform(45, 70)
        case AnomalyTypes.BATTERY_SOC:
            df.loc[idx, anomaly_type.value] = np.random.uniform(5, 25)
        case AnomalyTypes.TANK_PRESSURE:
            if np.random.rand() > 0.5:
                df.loc[idx, anomaly_type.value] = np.random.uniform(40, 80)
            else:
                df.loc[idx, anomaly_type.value] = np.random.uniform(160, 220)
        case AnomalyTypes.STRUCTURAL_INTEGRITY:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.CRITICAL_MODULES_STATUS:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.TELEMETRY_LINK_STATUS:
            df.loc[idx, anomaly_type.value] = 0


df.to_csv("telemetria_simulada.csv", index=False)
df.head()

,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,tank_pressure_bar,estimated_autonomy_min,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly,energy_available_kwh
0,2026-03-23 14:58:24.908417,29.071324,23.806065,46.778555,45.191906,90.382707,82.138615,8.419582,7.621775,144.254198,134.384321,0,1,1,True,3.472798
1,2026-03-23 14:59:24.908417,22.597107,6.025472,49.611182,100.365070,87.176299,88.676319,18.157670,7.292855,114.324446,161.830901,1,1,1,False,3.835179
2,2026-03-23 15:00:24.908417,24.041376,21.722864,46.060081,46.381963,66.743420,102.103742,21.036206,6.362924,136.092789,175.426394,1,1,1,False,3.138881
3,2026-03-23 15:01:24.908417,30.688255,3.478989,47.437638,29.989334,60.641828,80.681977,21.645799,2.981559,97.550657,131.467418,1,1,1,False,2.320983
4,2026-03-23 15:02:24.908417,25.136068,11.916168,50.200358,30.105437,95.776196,82.722286,17.693249,7.262421,99.666325,96.897498,1,1,1,False,3.977287


## 3. Verificação Pre-decolagem
Realiza a verificação de segurança utilizando os dados simulados.

In [13]:
from verificacao_pre_decolagem import pronto_para_decolar

df_result = df.copy()

# Itera por cada linha do dataframe
for idx, row in df.iterrows():
    df_result.loc[idx, "is_ready_to_launch"] = pronto_para_decolar(row.to_dict())

df_result.to_csv("telemetria_resultado.csv", index=False)
df_result.head()

,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,tank_pressure_bar,estimated_autonomy_min,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly,energy_available_kwh,is_ready_to_launch
0,2026-03-23 14:58:24.908417,29.071324,23.806065,46.778555,45.191906,90.382707,82.138615,8.419582,7.621775,144.254198,134.384321,0,1,1,True,3.472798,False
1,2026-03-23 14:59:24.908417,22.597107,6.025472,49.611182,100.365070,87.176299,88.676319,18.157670,7.292855,114.324446,161.830901,1,1,1,False,3.835179,True
2,2026-03-23 15:00:24.908417,24.041376,21.722864,46.060081,46.381963,66.743420,102.103742,21.036206,6.362924,136.092789,175.426394,1,1,1,False,3.138881,True
3,2026-03-23 15:01:24.908417,30.688255,3.478989,47.437638,29.989334,60.641828,80.681977,21.645799,2.981559,97.550657,131.467418,1,1,1,False,2.320983,True
4,2026-03-23 15:02:24.908417,25.136068,11.916168,50.200358,30.105437,95.776196,82.722286,17.693249,7.262421,99.666325,96.897498,1,1,1,False,3.977287,True


# 4. Teste do algoritmo de verificação de pré-decolagem
Visto que os dados com anomalia estão identificados com a flag `is_anomaly`, podemos utilizá-la para assegurar que todos os casos de anomalia foram capturados pelo algorítmo de verificação.

A verificação é feita da seguinte forma:
1. Filtra-se o dataframe de resultados por linhas onde `is_anomaly` é verdadeiro e `is_ready_to_launch` é verdadeiro.
2. Verifica-se se o dataframe filtrado está vazio indicando que não há possibilidade de lançamento em casos de anomalia.
3. Realiza a verificação oposta filtrando por linhas onde `is_anomaly` é falso e `is_ready_to_launch` é falso.
4. Verifica se o dataframe resultante está vazio indicando que não há possibilidade de abortar um lançamento quando não há anomalias.

In [14]:
# Filtra o dataframe por linhas com anomalia e prontas para decolagem
df_anomalies = df_result[(df_result["is_anomaly"] == True) & (df_result["is_ready_to_launch"] == True)]

# Assegura que o número de decolagens permitidas em dados com anomalia é igual a 0
assert(len(df_anomalies) == 0)

# Filtra o dataframe por linhas sem anomalia e com decolagem abortada
df_no_anomalies = df_result[(df_result["is_anomaly"] == False) & (df_result["is_ready_to_launch"] == False)]

# Assegura que o número de decolagens abortadas em dados sem anomalia é igual a 0
assert(len(df_no_anomalies) == 0)


# 5. Apresentação dos resultados
Exibe alguns exemplos de verificações pré-decolagens

In [ ]:
# Seleciona 5 dados com anomalia e 5 Sem anomalia
df_anomalies = df[df["is_anomaly"] == True]
df_non_anomalies = df[df["is_anomaly"] == False]
anomaly_samples = df_anomalies.sample(5)
non_anomaly_samples = df.sample(5)

# Agrega os dois dataframes
samples = pd.concat([anomaly_samples, non_anomaly_samples])

# Verifica a possibilidade de decolagem e imprime o resultado
for _, sample in samples.iterrows():
    if(pronto_para_decolar(sample.to_dict())):
        print(f"{sample.timestamp} - PRONTO PARA DECOLAR")
    else:
        print(f"{sample.timestamp} - DECOLAGEM ABORTADA")

2026-03-23 20:23:24.908417 - DECOLAGEM ABORTADA!
2026-03-23 17:44:24.908417 - DECOLAGEM ABORTADA!
2026-03-23 15:29:24.908417 - DECOLAGEM ABORTADA!
2026-03-23 14:58:24.908417 - DECOLAGEM ABORTADA!
2026-03-23 22:03:24.908417 - DECOLAGEM ABORTADA!
2026-03-23 21:19:24.908417 - PRONTO PARA DECOLAR!
2026-03-23 22:17:24.908417 - PRONTO PARA DECOLAR!
2026-03-23 17:51:24.908417 - PRONTO PARA DECOLAR!
2026-03-23 18:33:24.908417 - PRONTO PARA DECOLAR!
2026-03-23 16:26:24.908417 - PRONTO PARA DECOLAR!
